# 02b · Finding a better ResNet-50 (V2): step-by-step ablation on fold 0

Each step adds **one** change to the best configuration so far. Decisions use fold 0's **validation** set only;
the test fold is evaluated for the paper's ablation table but is **never** used to decide anything.

**Keep rule.** On this dataset validation accuracy saturates at 1.0 for every configuration, so macro-F1 alone
cannot rank them. A change is kept if it raises validation macro-F1 by more than 0.002, **or** macro-F1 is tied
(within 0.002) and it lowers the *calibrated* validation loss (NLL after temperature scaling, so label smoothing
cannot bias it) by at least 20 %. Every run stops early after 8 epochs without improvement, and the checkpoint is
the best validation macro-F1 with ties broken by lower validation loss.

| Step | Change |
|---|---|
| A1 | 224x224 input (upsampled) + backend head `Dropout(0.4)+Linear`, budget of up to 25 x 6,000 samples, checkpoint by val macro-F1 with loss tie-break |
| A2 | class-balanced sampling (each class, and each source patch within a class, equally likely) |
| A3 | leaf-patch augmentation (8 rotations/flips, random resized crop, brightness/contrast/saturation jitter (no hue: colour is diagnostic), random erasing) instead of AutoAugment |
| A4 | optimisation: AdamW lr 1e-3 head / 2e-4 backbone, wd 0.01, warm-up + cosine per step, label smoothing 0.1, weight EMA 0.999 |
| A5 | MixUp / CutMix (on half the batches) |

Prerequisite: 02a (R0, fold 0) is the starting point of the comparison.
Time: about 8-15 min per step (≈ 1 h).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import copy
import json

import numpy as np
import pandas as pd
import torch

from common import config, data, env
from common.cv import preds_path
from common.metrics import classification_metrics, fit_temperature
from common.train import TrainConfig, fit, predict, save_preds

config.ensure_dirs()
device = env.require_cuda()
env.seed_everything()
uniq, splits = data.load_unique(), data.load_splits()
images = data.load_images(uniq)
FOLD = 0
idx = data.fold_indices(uniq, splits, FOLD)
MIN_GAIN = 0.002      # macro-F1 gain that counts as better
NLL_GAIN = 0.20       # relative drop in calibrated val NLL that counts as better when macro-F1 ties

## Starting point: R0 on fold 0 (from notebook 02a)

In [ ]:
def scores_from_preds(name):
    out = {}
    for split in ("val", "test"):
        p = pd.read_csv(preds_path(name, FOLD, split))
        lg = p[[f"logit_{c}" for c in range(len(config.CLASSES))]].to_numpy()
        out[split] = classification_metrics(p["y_true"].to_numpy(), lg.argmax(1))
        if split == "val":
            y = p["y_true"].to_numpy()
            T = fit_temperature(lg, y)
            out["val_nll_cal"] = float(torch.nn.functional.cross_entropy(torch.tensor(lg / T), torch.tensor(y)))
    return out

r0_file = preds_path("R0", FOLD, "val")
if not r0_file.exists():
    raise FileNotFoundError("Run 02a_resnet50_current_recipe.ipynb first (needs R0 fold 0).")
r0 = scores_from_preds("R0")
print(f"R0 fold {FOLD}: val macro-F1 {r0['val']['macro_f1']:.4f} (calibrated NLL {r0['val_nll_cal']:.4f}) | "
      f"test macro-F1 {r0['test']['macro_f1']:.4f}")

## The steps

In [ ]:
A1 = TrainConfig(name="A1", arch="resnet50_dropout", size=224, train_aug="autoaugment", sampler="copies",
                 samples_per_epoch=6000, epochs=25, batch_size=64, optimizer="adamw", lr=3e-4, weight_decay=1e-4,
                 backbone_lr_mult=1.0, scheduler="cosine_epoch", select="val_macro_f1", tie_break_loss=True,
                 patience=8, notes="224 input + backend head")
STEPS = [
    ("A1", "224 px input + Dropout(0.4) head", {}),
    ("A2", "+ class-balanced sampling", {"sampler": "balanced"}),
    ("A3", "+ leaf-patch augmentation", {"train_aug": "leaf"}),
    ("A4", "+ optimisation recipe", {"lr": 1e-3, "backbone_lr_mult": 0.2, "weight_decay": 0.01,
                                     "scheduler": "cosine_warmup", "warmup_epochs": 1.0, "label_smoothing": 0.1,
                                     "ema_decay": 0.999}),
    ("A5", "+ MixUp/CutMix", {"mixup": True}),
]

eval_cache = {}
def eval_loader(split, size):
    key = (split, size)
    if key not in eval_cache:
        eval_cache[key] = data.make_loader(images, uniq, idx[split], data.build_transform("eval", size),
                                           batch_size=128, train=False, workers=0)
    return eval_cache[key]

rows = [{"step": "R0", "change": "current recipe (02a)", "kept": True,
         "val_macro_f1": r0["val"]["macro_f1"], "val_nll_cal": r0["val_nll_cal"], "val_acc": r0["val"]["accuracy"],
         "test_macro_f1": r0["test"]["macro_f1"], "test_acc": r0["test"]["accuracy"]}]
best_cfg, best_val, best_nll = None, r0["val"]["macro_f1"], r0["val_nll_cal"]


def better(f1, nll):
    if f1 > best_val + MIN_GAIN:
        return True
    return abs(f1 - best_val) <= MIN_GAIN and nll < (1 - NLL_GAIN) * best_nll
for step, desc, change in STEPS:
    cfg = copy.deepcopy(A1 if best_cfg is None else best_cfg)
    for k, v in change.items():
        setattr(cfg, k, v)
    cfg.name, cfg.notes = step, desc
    done = config.RUNS_DIR / "ablation" / step / "history.json"
    if done.exists() and preds_path(f"abl_{step}", FOLD, "test").exists():
        print(f"{step}: already done, re-using")
    else:
        tl = data.make_loader(images, uniq, idx["train"], data.build_transform(cfg.train_aug, cfg.size),
                              batch_size=cfg.batch_size, train=True, sampler_mode=cfg.sampler,
                              samples_per_epoch=cfg.samples_per_epoch, workers=cfg.workers, seed=config.SEED)
        res = fit(cfg, tl, eval_loader("val", cfg.size), device, config.RUNS_DIR / "ablation" / step)
        for split in ("val", "test"):
            lg, y, j = predict(res["model"], eval_loader(split, cfg.size), device)
            save_preds(preds_path(f"abl_{step}", FOLD, split), uniq, lg, y, j)
        del tl, res
        torch.cuda.empty_cache()
    sc = scores_from_preds(f"abl_{step}")
    # A1 changes the setup itself (input size, head, budget), so it always becomes the base for the next steps
    kept = step == "A1" or better(sc["val"]["macro_f1"], sc["val_nll_cal"])
    print(f"{step} {desc}: val macro-F1 {sc['val']['macro_f1']:.4f}, calibrated NLL {sc['val_nll_cal']:.4f} "
          f"(best so far {best_val:.4f} / {best_nll:.4f}) -> {'KEEP' if kept else 'drop'}")
    if kept:
        best_cfg, best_val, best_nll = cfg, sc["val"]["macro_f1"], sc["val_nll_cal"]
    rows.append({"step": step, "change": desc, "kept": kept, "val_macro_f1": sc["val"]["macro_f1"],
                 "val_nll_cal": sc["val_nll_cal"],
                 "val_acc": sc["val"]["accuracy"], "test_macro_f1": sc["test"]["macro_f1"],
                 "test_acc": sc["test"]["accuracy"]})

ablation = pd.DataFrame(rows)
ablation

## Chosen V2 configuration
Same settings as the chosen step, including patience 8 and the loss tie-break.

In [ ]:
v2 = copy.deepcopy(best_cfg)
v2.name = "V2"
v2.patience = 8
v2.notes = "Chosen by 02b: " + ", ".join(r["change"] for r in rows[1:] if r["kept"])
env.write_json(config.CNN_RESULTS / "v2_config.json", v2.to_dict())
env.write_json(config.CNN_RESULTS / "v2_ablation.json",
               {"fold": FOLD, "min_gain": MIN_GAIN, "nll_gain": NLL_GAIN, "rows": rows, "meta": env.run_meta()})
print(json.dumps(v2.to_dict(), indent=2))